#  ⭐ `dim_atc`


In [1]:
%run ../../config/bootstrap.py


from pathlib import Path
from utils import get_project_root 

import re
import pandas as pd
from unidecode import unidecode

In [2]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
bronze = pd.read_excel(Path(project_root) / "data/01_bronze/atc/2025 ATC index with DDDs_english.xlsx")  

In [4]:
bronze.head()

,ATC code,ATC level name,DDD,Unit,Adm.R,Comment
0,A,ALIMENTARY TRACT AND METABOLISM,NaN,NaN,NaN,NaN
1,A01,STOMATOLOGICAL PREPARATIONS,NaN,NaN,NaN,NaN
2,A01AA,Caries prophylactic agents,NaN,NaN,NaN,NaN
3,A01AA01,sodium fluoride,1.1,mg,O,0.5 mg fluoride
4,A01AA02,sodium monofluorophosphate,NaN,NaN,NaN,NaN


In [4]:
bronze.DETENTOR_REGISTRO.nunique()

10367

In [5]:
bronze.to_csv(Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos_DETENTOR_REGISTRO.csv", sep=";", index=False)

# 🥈 Silver

normalized data, medium quality


In [6]:
silver = pd.read_csv(Path(project_root) / "data/02_silver/hist_dim_detentor_registro/MANUAL_detentor_registro.csv", sep=";") ## Manual Normalization
 
silver.columns

Index(['DETENTOR_REGISTRO', 'DETENTOR_REGISTRO_PADRONIZADO'], dtype='object')

In [7]:
silver.head(2)

,DETENTOR_REGISTRO,DETENTOR_REGISTRO_PADRONIZADO
0,ABACUS MEDICINE,ABACUS MEDICINE
1,ABBOLT,ABBOT


In [8]:
silver.DETENTOR_REGISTRO.nunique()

6675

In [9]:
silver.DETENTOR_REGISTRO_PADRONIZADO.nunique()

271

In [10]:
silver.DETENTOR_REGISTRO_PADRONIZADO.value_counts().head(20)

DETENTOR_REGISTRO_PADRONIZADO
DESCONHECIDO              2080
ASTRAZENECA                284
PFIZER                     257
FIOCRUZ                    195
CRISTALIA                  176
JOHNSON & JOHNSON          161
BLAU FARMACÊUTICA          109
EMS                        103
UNIAO QUIMICA               97
MEDICAMENTO MANIPULADO      97
SINOVAC BIONTECH            94
SANOFI                      93
EUROFARMA                   87
ABL                         82
TEUTO                       79
HIPOLABOR                   76
HALEX ISTAR                 68
MERCK                       67
FRESENIUS KABI              60
INSTITUTO BUTANTAN          59
Name: count, dtype: int64

In [11]:
dim = silver[['DETENTOR_REGISTRO']].drop_duplicates().fillna('DESCONHECIDO')
dim.head(20)

,DETENTOR_REGISTRO
0,ABACUS MEDICINE
1,ABBOLT
2,ABBOT
3,ABBOT BRASIL
4,ABBOT L: 0071/20 V: 31/01/2022
5,ABBOT L: 1040818 V: 31/03/2022
6,ABBOT LABORATORIUOS DO BRASIL LTDA
7,ABBOTH BIOLOGICALS
8,ABBOTT
9,ABBOTT B.V.


In [12]:
# termos genéricos que não diferenciam empresas farmacêuticas
STOPWORDS_EMPRESA = {
    "SA", "S/A", "S.A", "S.A.", "SA.", "S A",
    "LTDA", "LTD", "EIRELI", "ME", "EPP",
    "COMERCIO", "COMERCIAL", "INDUSTRIA", "INDUSTRIAL",
    "LABORATORIO", "LABORATORIOS", "LAB",
    "FARMACEUTICA", "FARMACEUTICO", "FARMACEUTICOS","PHARMA",
    "DO", "DA", "DE", "DOS", "DAS", "E",
}

def limpar_texto_empresa(texto: str) -> str:
    """Normaliza texto de nome de empresa para criar uma chave estável."""
    if pd.isna(texto):
        return ""
    
    s = str(texto).strip()
    s = unidecode(s)           # remove acentos
    s = s.upper()              # caixa alta
    s = re.sub(r"[^A-Z0-9 ]+", " ", s)  # tira pontuação
    tokens = [t for t in s.split() if t not in STOPWORDS_EMPRESA]
    s = " ".join(tokens)
    s = re.sub(r"\s+", " ", s).strip()  # colapsa espaços
    
    return s

def criar_key_detentor(texto: str) -> str:
    """Wrapper pra ficar semântico."""
    return limpar_texto_empresa(texto)


In [13]:
dim["DETENTOR_REGISTRO_VALOR"] = dim["DETENTOR_REGISTRO"].apply(criar_key_detentor)
dim["DETENTOR_REGISTRO_VALOR"].nunique()


5404

In [14]:
silver[['DETENTOR_REGISTRO_PADRONIZADO']].nunique()

DETENTOR_REGISTRO_PADRONIZADO    271
dtype: int64

In [15]:
dim = silver[['DETENTOR_REGISTRO_PADRONIZADO']].drop_duplicates().fillna('DESCONHECIDO')
dim["DETENTOR_REGISTRO_VALOR"] = dim["DETENTOR_REGISTRO_PADRONIZADO"].apply(criar_key_detentor)
dim["DETENTOR_REGISTRO_VALOR"].nunique()


265

In [16]:
dim.head()

,DETENTOR_REGISTRO_PADRONIZADO,DETENTOR_REGISTRO_VALOR
0,ABACUS MEDICINE,ABACUS MEDICINE
1,ABBOT,ABBOT
30,ABBVIE,ABBVIE
46,ABL,ABL
128,ACCORD FARMACÊUTICA,ACCORD


In [17]:
freq = (
    dim
    .groupby(["DETENTOR_REGISTRO_VALOR", "DETENTOR_REGISTRO_PADRONIZADO"])
    .size()
    .reset_index(name="n")
)
freq.sort_values("n", ascending=False).head(10)

,DETENTOR_REGISTRO_VALOR,DETENTOR_REGISTRO_PADRONIZADO,n
0,ABACUS MEDICINE,ABACUS MEDICINE,1
1,ABBOT,ABBOT,1
2,ABBVIE,ABBVIE,1
3,ABL,ABL,1
4,ACCORD,ACCORD FARMACÊUTICA,1
5,ACHE,ACHÉ,1
6,ADC THERAPEUTICS,ADC THERAPEUTICS,1
7,ADIUM,ADIUM,1
8,AIRELA,AIRELA,1
9,ALCON,ALCON,1


In [18]:
dim_detentor = (
    freq
    .sort_values(["DETENTOR_REGISTRO_VALOR", "n"], ascending=[True, False])
    .drop_duplicates("DETENTOR_REGISTRO_VALOR")
    #.rename(columns={"DETENTOR_REGISTRO_PADRONIZADO": "DETENTOR_REGISTRO_CANONICO"})
    [["DETENTOR_REGISTRO_VALOR", "DETENTOR_REGISTRO_PADRONIZADO"]]
    .reset_index(drop=True)
)
dim_detentor.head()

,DETENTOR_REGISTRO_VALOR,DETENTOR_REGISTRO_PADRONIZADO
0,ABACUS MEDICINE,ABACUS MEDICINE
1,ABBOT,ABBOT
2,ABBVIE,ABBVIE
3,ABL,ABL
4,ACCORD,ACCORD FARMACÊUTICA


In [19]:
dim_detentor.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   DETENTOR_REGISTRO_VALOR        265 non-null    object
 1   DETENTOR_REGISTRO_PADRONIZADO  265 non-null    object
dtypes: object(2)
memory usage: 4.3+ KB


In [20]:
silver[['DETENTOR_REGISTRO_PADRONIZADO']].nunique()

DETENTOR_REGISTRO_PADRONIZADO    271
dtype: int64

In [21]:
dim_detentor =  silver[['DETENTOR_REGISTRO_PADRONIZADO']].drop_duplicates().fillna('DESCONHECIDO')
dim_detentor["DETENTOR_REGISTRO_VALOR"] = dim_detentor["DETENTOR_REGISTRO_PADRONIZADO"].apply(criar_key_detentor)
dim_detentor.insert(0, "DETENTOR_REGISTRO_CHAVE", range(1, len(dim_detentor) + 1))
dim_detentor.info()


<class 'pandas.core.frame.DataFrame'>
Index: 271 entries, 0 to 6764
Data columns (total 3 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   DETENTOR_REGISTRO_CHAVE        271 non-null    int64 
 1   DETENTOR_REGISTRO_PADRONIZADO  271 non-null    object
 2   DETENTOR_REGISTRO_VALOR        271 non-null    object
dtypes: int64(1), object(2)
memory usage: 8.5+ KB


In [22]:
dim_detentor.DETENTOR_REGISTRO_CHAVE.nunique()

271

In [23]:
hist = silver.merge(dim_detentor, on="DETENTOR_REGISTRO_PADRONIZADO", how="left")
hist.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6765 entries, 0 to 6764
Data columns (total 4 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   DETENTOR_REGISTRO              6761 non-null   object
 1   DETENTOR_REGISTRO_PADRONIZADO  6765 non-null   object
 2   DETENTOR_REGISTRO_CHAVE        6765 non-null   int64 
 3   DETENTOR_REGISTRO_VALOR        6765 non-null   object
dtypes: int64(1), object(3)
memory usage: 211.5+ KB


In [24]:
hist.head()

,DETENTOR_REGISTRO,DETENTOR_REGISTRO_PADRONIZADO,DETENTOR_REGISTRO_CHAVE,DETENTOR_REGISTRO_VALOR
0,ABACUS MEDICINE,ABACUS MEDICINE,1,ABACUS MEDICINE
1,ABBOLT,ABBOT,2,ABBOT
2,ABBOT,ABBOT,2,ABBOT
3,ABBOT BRASIL,ABBOT,2,ABBOT
4,ABBOT L: 0071/20 V: 31/01/2022,ABBOT,2,ABBOT


In [25]:
dim_detentor.DETENTOR_REGISTRO_PADRONIZADO.nunique()

271

In [26]:
dim_detentor.DETENTOR_REGISTRO_CHAVE.nunique()

271

In [27]:
hist.to_parquet(Path(project_root) / "data/02_silver/hist_dim_detentor_registro/hist_dim_detentor_registro.parquet", index=False)

# 🥇 Gold


In [28]:
hist.columns

Index(['DETENTOR_REGISTRO', 'DETENTOR_REGISTRO_PADRONIZADO',
       'DETENTOR_REGISTRO_CHAVE', 'DETENTOR_REGISTRO_VALOR'],
      dtype='object')

In [29]:
dim = hist[['DETENTOR_REGISTRO_CHAVE', 'DETENTOR_REGISTRO_VALOR']].drop_duplicates()
dim.to_parquet(Path(project_root) / "data/03_gold/dim_detentor_registro/dim_detentor_registro.parquet", index=False)

In [31]:

dim.to_csv(Path(project_root) / "data/03_gold/dim_detentor_registro/dim_detentor_registro.csv", sep=";", index=False)